# Meta-Analysis of RAG Output

## Libraries Imported

In [ ]:
import pandas as pd
import json
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Import and Transformation

### Data Import

In [ ]:
# load the responses from the JSON file
with open('./latest_output.json', 'r') as f:
    responses = json.load(f)


### Structuring the JSON File

In [ ]:
def parse_q_string(q_string):
    parts = {}
    for field in ["QuestionText", "ShortAnswer", "Reasoning", "Evidence"]:
        # Match the value starting at the label and ending at the next label or end of string
        pattern = rf"{field}:\s*(.*?)(?=(QuestionText:|ShortAnswer:|Reasoning:|Evidence:|$))"
        match = re.search(pattern, q_string, re.DOTALL)
        if match:
            parts[field] = match.group(1).strip().rstrip(",")  # remove trailing comma if any
        else:
            parts[field] = 'Unfilled'
    return parts

In [ ]:
for key in responses.keys():
    for question, answer in responses[key]['answers'].items():
        if type(answer) == str:
            responses[key]['answers'][question] = parse_q_string(answer)

In [ ]:
for key in responses.keys():
    for question, answer in responses[key]['answers'].items():
        if type(answer) == str:
            responses = parse_q_string(answer)
            print("k:")
            print(k)
            print(f"id: {key} and title {responses[key]['metadata']['Title']}")
            break

### Flattening JSON File

In [ ]:
def flattenDict(d, parent_key='', sep='-'):
    """
    Flattens a nested dictionary.

    Args:
        d (dict): The nested dictionary to flatten.
        parent_key (str): The parent key to use for the flattened dictionary.
        sep (str): The separator to use for the flattened dictionary.

    Returns:
        dict: The flattened dictionary.
    """
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flattenDict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)

def flattenJson(json):
    """
    Flattens a response json.

    Args:
        json (dict): The response json to flatten.

    Returns:
        dict: The flattened response json.
    """
    newJson={}
    for k,v in json.items():
        newJson[k]={}
        for k2,v2 in v.items():
            if isinstance(v2,dict):
                newJson[k].update(flattenDict(v2))
            else:
                newJson[k][k2]=v2
    return newJson

In [ ]:
flat_responses=flattenJson(responses)
flat_responses['1']

### Converting JSON File to Pandas Dataframe

In [ ]:
# convert the json to a pandas dataframe
df = pd.DataFrame.from_dict(flat_responses, orient='index')
df.set_index('paper_id', inplace=True)

### Using Evidence to Encode Unsure/Not Provided Columns

In [ ]:
# Encoded Answers for Questions
real_encodings = {
    'q1': [],
    'q2': [],
    'q3': [],
    'q4': [],
    'q5': [],
    'q6': [],
    'q7': [],
    'q8': [],
    'q9': [],
    'q10': [],
    'q11': [],
    'q12': [],
    'q13': [],
    'q14': [],
    'q15': [],
    'q16': [],
    'q17': [],
    'q18': [],
    'q19': [],
    'q20': [],
    'q21': [],
    'q22': [],
    'q23': [],
    'q24': [],
    'q25': []
}

In [ ]:
# filter out record where "Key" is ""
# cited_df = df[df["Key"] != ""]
cited_df = df

## Distribution of Values for Each Question

In [ ]:
general_df = cited_df

# Check if any column contains lists
for col in general_df.columns:
    if general_df[col].apply(lambda x: isinstance(x, list)).any():
        def safe_join(x):
            if isinstance(x, list):
                try:
                    return ", ".join(str(item) for item in x)
                except Exception as e:
                    print(f"Error in column '{col}': {x} ({type(x)})")
                    raise
            return x

        general_df[col] = general_df[col].apply(safe_join)

In [ ]:
for i, row in general_df.iterrows():
    nan_in_row = row.isna()
    columns_with_nan = nan_in_row[nan_in_row].index.tolist()
    
    if columns_with_nan:  # only print rows that actually have NaNs
        print(f"Row {i} has NaN in columns: {columns_with_nan}")
        for k in row:
            print(k)
    break

In [ ]:
print(general_df)

In [ ]:
# plot distribution of values in "Q1-ShortAnswer" column

i = 1
for q in range(i, 27):
    questionText = general_df[f"Q{q}-QuestionText"].iloc[0]
    print(f"Q{q}: {questionText}")
    
    counts = general_df[f"Q{q}-ShortAnswer"].value_counts(dropna=False)  # include NaN if needed    
    
    plt.figure(figsize=(6, 4))
    counts.plot(kind='bar')
    plt.title(f'Distribution of answers for Q{q}: {questionText}')
    plt.xlabel('Answer')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    

## Trends within Task Type in Machine Learning

In [ ]:
allCategories = cited_df[["Q16-ShortAnswer","Published Year", "Q24-ShortAnswer", "Q10-ShortAnswer", "Q21-ShortAnswer", "Q23-ShortAnswer", "Q20-ShortAnswer", "Q15-ShortAnswer"]]
allCategories.columns = ["Task Type","Year", "Architecture", "Benchmark", "Uncertainty", "OOD", "Encoding", "Learning Paradigm"]
allCategories=allCategories.reset_index(drop=True)

# exclude "Not Applicable" and "Unsure"
allCategories = allCategories[allCategories["Task Type"] != "Not provided"]
allCategories = allCategories[allCategories["Task Type"] != "Unsure"]
allCategories = allCategories[allCategories["Architecture"] != "Not provided"]
allCategories = allCategories[allCategories["Architecture"] != "Unsure"]
allCategories = allCategories[allCategories["Benchmark"] != "Not provided"]
allCategories = allCategories[allCategories["Uncertainty"] != "Unsure"]
allCategories = allCategories[allCategories["OOD"] == "Yes"]
allCategories = allCategories[allCategories["Encoding"] != "Not provided"]

In [ ]:
allCategories

### Distribution of Task Type Over all Papers

In [ ]:
freq = allCategories.loc[:,"Task Type"].value_counts().sort_values(ascending=False)
plt.figure(figsize=(5, 3))
plt.bar(freq.index, freq.values)
plt.xlabel("Task Type")
plt.ylabel("Number of Studies")
plt.title("Distribution of Task Types")
plt.xticks(rotation=30, ha='right')  # Rotates the x-tick labels by 30 degrees
plt.show()

### Distribution of Task Type Over all Papers Over Time

In [ ]:
# Group by 'Year' and 'Task Type' to count the number of occurrences for each combination
category_trend = allCategories.groupby(['Year', 'Task Type']).size().reset_index(name='Count')

# Pivot the table so that each category becomes a column and fill missing values with 0
pivot_table = category_trend.pivot(index='Year', columns='Task Type', values='Count').fillna(0)

# Ensure the data is sorted by Year
pivot_table.sort_index(inplace=True)

# Sort the columns by total frequency (sum over years) in descending order
total_counts = pivot_table.sum(axis=0)
sorted_categories = total_counts.sort_values(ascending=False).index
pivot_table = pivot_table[sorted_categories]

# Prepare the data for the stackplot
years = pivot_table.index.values
stack_data = [pivot_table[col].values for col in pivot_table.columns]

# Generate a pastel color palette using the 'tab10' colormap
cmap = plt.get_cmap("tab10")
colors = [cmap(i) for i in np.linspace(0, 1, len(pivot_table.columns))]

plt.figure(figsize=(12, 3))
plt.stackplot(years, stack_data, labels=pivot_table.columns, colors=colors)

plt.xlabel("Year")
plt.ylabel("Count")
plt.title("Trend of Task Types Over Time")
plt.legend(title="Task Type", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Assuming pivot_table is defined and sorted by total frequency as before
years = pivot_table.index.values
stack_data = [pivot_table[col].values for col in pivot_table.columns]

# Compute the cumulative sum for stacking.
cumulative = np.cumsum(stack_data, axis=0)

# Generate a pastel color palette using the 'Pastel1' colormap.
cmap = plt.get_cmap("tab10")
colors = [cmap(i) for i in np.linspace(0, 1, len(pivot_table.columns))]

# Define a list of hatch patterns with increased density.
hatches = ['///', '\\\\\\', '|||', '---', '+++', 'xxx', 'ooo', 'OOO', '...', '***']

plt.figure(figsize=(12, 4))

# Plot the first category from 0 to the first cumulative sum.
plt.fill_between(
    years, 
    0, 
    cumulative[0], 
    facecolor=colors[0], 
    edgecolor='white', 
    hatch=hatches[0],
    label=pivot_table.columns[0]
)

# For each subsequent category, fill between the previous and current cumulative values.
for i in range(1, len(stack_data)):
    plt.fill_between(
        years, 
        cumulative[i-1], 
        cumulative[i], 
        facecolor=colors[i], 
        edgecolor='white', 
        hatch=hatches[i % len(hatches)],
        label=pivot_table.columns[i]
    )

plt.xlabel("Year")
plt.ylabel("Count")
plt.title("Trend of Categories Over Time with Colors and Dense Patterns")
plt.legend(title="Category", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


### Distribution of Learning Pardigm per Task Type

In [ ]:
learning_counts = allCategories.groupby(['Learning Paradigm', 'Task Type']).size().reset_index(name='Count')

plt.figure(figsize=(14, 8))
sns.barplot(data=learning_counts, x='Count', y='Task Type', hue='Learning Paradigm')
plt.title("Learning Paradigm Usage per Task Type")
plt.tight_layout()
plt.show()

### Distribution of Architecture per Task Type

In [ ]:
arch_counts = allCategories.groupby(['Architecture', 'Task Type']).size().reset_index(name='Count')

plt.figure(figsize=(14, 8))
sns.barplot(data=arch_counts, x='Count', y='Task Type', hue='Architecture')
plt.title("Architecture Usage per Task Type")
plt.tight_layout()
plt.show()

### Distribution of Benchmarks used per Task Type

In [ ]:
bench_counts = allCategories.groupby(['Benchmark', 'Task Type']).size().reset_index(name='Count')

plt.figure(figsize=(14, 8))
sns.barplot(data=bench_counts, x='Count', y='Task Type', hue='Benchmark')
plt.title("Benchmark Usage per Task Type")
plt.tight_layout()
plt.show()

### Distribution of Techniques that Evaluate Uncertainty per Task Type

In [ ]:
allCategories = allCategories[allCategories['Uncertainty'] == 'Yes']

uncertainty_counts = allCategories.groupby(['Uncertainty', 'Task Type']).size().reset_index(name='Count')

plt.figure(figsize=(14, 8))
sns.barplot(data=uncertainty_counts, x='Count', y='Task Type', hue='Uncertainty')
plt.title("Uncertainty Usage per Task Type")
plt.tight_layout()
plt.show()

### Distribution of Techniques that Evaluate Out of Distribution per Task Type

In [ ]:
ood_counts = allCategories.groupby(['OOD', 'Task Type']).size().reset_index(name='Count')

plt.figure(figsize=(14, 8))
sns.barplot(data=ood_counts, x='Count', y='Task Type', hue='OOD')
plt.title("OOD Usage per Task Type")
plt.tight_layout()
plt.show()

### Distribution of Encoding Techniques per Task Type

In [ ]:
encoding_counts = allCategories.groupby(['Encoding', 'Task Type']).size().reset_index(name='Count')

plt.figure(figsize=(14, 8))
sns.barplot(data=encoding_counts, x='Count', y='Task Type', hue='Encoding')
plt.title("Encoding Usage per Task Type")
plt.tight_layout()
plt.show()

## Trends in Dataset Used

In [ ]:
df = cited_df[['Published Year', 'Q10-ShortAnswer']].dropna()
df['Q10-ShortAnswer'] = df['Q10-ShortAnswer'].apply(
    lambda x: [item.strip().upper() for item in x.split(',')]
)
df = df.explode('Q10-ShortAnswer').rename(columns={'Q10-ShortAnswer': 'Benchmark'})

### Distribution of Benchmarks used Over all Papers

In [ ]:
freq = df.loc[:,"Benchmark"].value_counts().sort_values(ascending=False)
plt.figure(figsize=(5, 3))
plt.bar(freq.index, freq.values)
plt.xlabel("Benchmark")
plt.ylabel("Number of Studies")
plt.title("Distribution of Benchmarks Used")
plt.xticks(rotation=30, ha='right')  # Rotates the x-tick labels by 30 degrees
plt.show()

### Distribution of Benchmarks used Over Time

In [ ]:
benchmark_trend = df.groupby(['Published Year', 'Benchmark']).size().reset_index(name='Count')
pivot_table = benchmark_trend.pivot(index='Published Year', columns='Benchmark', values='Count').fillna(0)


total_counts = pivot_table.sum(axis=0)
sorted_benchmarks = total_counts.sort_values(ascending=False).index
pivot_table = pivot_table[sorted_benchmarks]


years = pivot_table.index.values
stack_data = [pivot_table[benchmark].values for benchmark in pivot_table.columns]

top_n = 10
pivot_table = pivot_table[sorted_benchmarks[:top_n]]
stack_data = [pivot_table[col].values for col in pivot_table.columns]


cmap = plt.get_cmap("tab10")
colors = [cmap(i) for i in np.linspace(0, 1, len(pivot_table.columns))]

plt.figure(figsize=(12, 3))
plt.stackplot(pivot_table.index.values, stack_data, labels=pivot_table.columns, colors=colors)

plt.xlabel("Year")
plt.ylabel("Count")
plt.title("Trend of Benchmarks Over Time")
plt.legend(title="Benchmark", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()